# 4.9 · 支持向量回归 / Support Vector Regression (SVR)

> **课程定位 / Where this fits**
> **Part 4 第 9 课**。4.8 的非线性靠人工指定函数形式; SVR 用**核技巧**自动处理非线性, 且哲学独特：**只惩罚"管道外"的点 (ε-不敏感损失)**, 管道内的误差完全不管。它是 SVM(Part 5.5 分类) 的回归版, 核方法的代表。
> SVR uses the kernel trick for automatic nonlinearity, and only penalizes points outside an epsilon-tube. The regression sibling of SVM.

> 💡 **面试相关 / Interview-relevant**
> - "SVR 和线性回归损失函数区别" ★★★★（ε-不敏感 vs 平方）
> - "核技巧是什么" ★★★★★（隐式高维映射）
> - "支持向量是什么" ★★★★
> - "SVR 的 C, ε, gamma 各管什么" ★★★

---

## 学习目标 / Learning Objectives
1. 理解 **ε-不敏感损失**：管道内零惩罚, 管道外线性惩罚。
2. 理解**核技巧**：不显式升维就能拟合非线性 (RBF 核)。
3. 理解**支持向量**：只有管道边界上/外的点决定模型。
4. 调 **C / ε / gamma** 三个关键超参。

## 目录 / TOC
1. [ε-不敏感损失: 独特哲学 ⭐](#1)
2. [核技巧: 自动非线性 ⭐](#2)
3. [数据 + 线性 vs RBF SVR](#3)
4. [支持向量: 谁决定模型](#4)
5. [C / ε / gamma 三超参](#5)
6. [对比其他回归](#6)
7. [小结](#7)


<a id="1"></a>
## 1. ε-不敏感损失: 独特哲学 ⭐ / ε-insensitive Loss

普通回归（OLS）惩罚**所有**误差（平方）。SVR 完全不同——画一条宽度 $2\epsilon$ 的"管道"绕着预测线, **管道内的点零惩罚**, 只惩罚管道外的部分（线性）：

$$L_\epsilon(y, \hat{y}) = \max(0, |y - \hat{y}| - \epsilon)$$

| 损失 | 管道内 ($|y-\hat{y}|\le\epsilon$) | 管道外 |
|---|---|---|
| OLS (平方) | 仍惩罚 $(y-\hat{y})^2$ | 平方(对大误差极敏感) |
| **ε-不敏感** | **零惩罚** | 线性(对异常值更稳健) |

**两个好处**：
1. **稀疏性**: 管道内的点对模型**毫无影响** → 只有边界上/外的点(=支持向量)定义模型
2. **稳健性**: 管道外用**线性**惩罚(不是平方) → 比 OLS 抗异常值(接 4.15)

SVR 的目标 = 最小化 $\frac{1}{2}\|\mathbf{w}\|^2 + C\sum L_\epsilon$（$\|\mathbf{w}\|^2$ 是 L2 正则 = 让管道尽量平坦, C 控制对管道外违规的容忍）。
SVR minimizes flatness (L2 on w) + C times tube violations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 非线性数据: sinc 函数 + 噪声 / nonlinear sinc data
x = np.sort(rng.uniform(-4, 4, 200))
y = 2*np.sin(x) + rng.normal(0, 0.2, 200)   # 强信号平滑非线性
X = x.reshape(-1, 1)
print("数据: 2·sin(x) + 噪声, 强非线性")


<a id="2"></a>
## 2. 核技巧: 自动非线性 ⭐ / The Kernel Trick

**核技巧是机器学习最优雅的思想之一**。线性 SVR 只能拟合直线。要拟合非线性, 4.3 的办法是显式造高阶特征 $[x, x^2, \dots]$——维度爆炸。

**核技巧**: 把数据隐式映射到极高维（甚至无限维）空间, 在那里做线性回归——**但永远不显式计算高维坐标**, 只需要**核函数** $K(\mathbf{x}_i, \mathbf{x}_j)$ 给出"两点在高维空间的内积":

$$K_{\text{RBF}}(\mathbf{x}_i, \mathbf{x}_j) = \exp\big(-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2\big)$$

**RBF 核对应无限维特征空间**, 却只需算欧氏距离。这就是"用有限计算做无限维线性回归"的魔法（0.7 节内积视角的高潮）。

| 核 | 公式 | 适合 |
|---|---|---|
| linear | $\mathbf{x}_i^\top\mathbf{x}_j$ | 线性关系 |
| poly | $(\gamma\mathbf{x}_i^\top\mathbf{x}_j + r)^d$ | 多项式关系 |
| **RBF** ⭐ | $e^{-\gamma\|\mathbf{x}_i-\mathbf{x}_j\|^2}$ | 通用非线性(默认) |

The kernel trick: implicitly map to infinite-dimensional space and do linear regression there, computing only inner products via the kernel — never the coordinates.


<a id="3"></a>
## 3. 数据 + 线性 vs RBF SVR / Linear vs RBF


In [ ]:
# SVR 对尺度敏感 → 标准化 (3.4) / scale-sensitive
Xs = StandardScaler().fit_transform(X)

x_plot = np.linspace(-4, 4, 300).reshape(-1, 1)
x_plot_s = StandardScaler().fit(X).transform(x_plot)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, y, alpha=0.4, s=15, label="数据")
for kernel, c in [("linear", "orange"), ("rbf", "red")]:
    svr = SVR(kernel=kernel, C=10, epsilon=0.1, gamma="scale").fit(Xs, y)
    ax.plot(x_plot, svr.predict(x_plot_s), color=c, lw=2, label=f"SVR ({kernel})")
ax.legend(); ax.set_title("线性 SVR 拟合不了 sinc; RBF 核 SVR 完美捕捉非线性")
plt.tight_layout(); plt.show()
print("linear 核: 只能画直线, 对 sinc 无能为力")
print("RBF 核: 自动拟合复杂非线性曲线 — 核技巧的威力, 无需人工指定函数形式(对比 4.8)")


<a id="4"></a>
## 4. 支持向量: 谁决定模型 / Support Vectors

**SVR 的稀疏性**: 模型只由**支持向量**（管道边界上/外的点）决定。管道**内**的点删掉模型不变——它们对预测毫无贡献。


In [ ]:
svr = SVR(kernel="rbf", C=10, epsilon=0.2, gamma="scale").fit(Xs, y)
sv_idx = svr.support_                        # 支持向量的索引 / support vector indices
print(f"总样本 {len(X)}, 支持向量 {len(sv_idx)} 个 ({len(sv_idx)/len(X):.0%})")
print(f"→ 只有 {len(sv_idx)} 个点定义了整个模型, 其余 {len(X)-len(sv_idx)} 个(管道内)删了也不变\n")

pred = svr.predict(Xs)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, y, alpha=0.3, s=15, label="管道内点 (无影响)")
ax.scatter(x[sv_idx], y[sv_idx], facecolor="none", edgecolor="red", s=60, label="支持向量")
ax.plot(x_plot, svr.predict(x_plot_s), "b-", lw=2, label="SVR 预测")
# 画 ε 管道 / draw the epsilon tube
ax.fill_between(x_plot.ravel(), svr.predict(x_plot_s)-0.2, svr.predict(x_plot_s)+0.2,
                alpha=0.15, color="blue", label="ε 管道 (±0.2)")
ax.legend(fontsize=8); ax.set_title("红圈=支持向量(边界上/外); 管道内点对模型零贡献")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. C / ε / gamma 三超参 / The Three Hyperparameters

| 超参 | 控制 | 大 → | 小 → |
|---|---|---|---|
| **C** | 对管道外违规的惩罚 | 拟合更紧(可能过拟合) | 更平滑(可能欠拟合) |
| **ε** | 管道宽度 | 管道宽, 支持向量少, 模型简单 | 管道窄, 支持向量多, 拟合紧 |
| **gamma** (RBF) | 每个点影响范围 | 影响范围小, 曲线扭曲(过拟合) | 影响范围大, 曲线平滑 |


In [ ]:
from sklearn.model_selection import GridSearchCV, KFold

# gamma 的影响 (过拟合 vs 欠拟合) / gamma effect
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, g in zip(axes, [0.01, 0.5, 50]):
    s = SVR(kernel="rbf", C=10, gamma=g).fit(Xs, y)
    ax.scatter(x, y, alpha=0.3, s=12)
    ax.plot(x_plot, s.predict(x_plot_s), "r-", lw=2)
    ax.set_title(f"gamma={g}\n{'欠拟合(太平滑)' if g<0.1 else ('刚好' if g<5 else '过拟合(太扭曲)')}")
plt.tight_layout(); plt.show()

# 网格搜索调参 / grid search
grid = GridSearchCV(SVR(kernel="rbf"),
                    {"C": [1, 10, 100], "gamma": [0.05, 0.2, 1, 5], "epsilon": [0.01, 0.05, 0.1]},
                    cv=KFold(5, shuffle=True, random_state=0), scoring="r2").fit(Xs, y)
print(f"最优超参: {grid.best_params_}")
print(f"最优 CV R² = {grid.best_score_:.3f}")

print("(注意: 数据按 x 排序, CV 必须 shuffle — 否则连续切分导致外推, 3.10 教训)")


<a id="6"></a>
## 6. 对比其他回归 / Comparison

| | SVR | 线性回归 | 树/森林 |
|---|---|---|---|
| 非线性 | ✅ 核技巧 | ❌ (需手工特征) | ✅ |
| 异常值 | 较稳健(ε+线性损失) | 敏感(平方) | 稳健 |
| 高维 | 较好 | 好 | 好 |
| 大样本 | ❌ **慢**($O(n^2)\sim O(n^3)$) | 快 | 中 |
| 可解释 | 差(核空间) | 好 | 中 |
| 缩放 | **必须**(3.4) | 正则时需要 | 不需要 |

**SVR 的定位**: 中小样本 + 强非线性 + 要稳健, 表现优异; 但**大样本下慢**(核矩阵 $O(n^2)$), 被树模型/神经网络取代。面试要知道它的核技巧思想（迁移到 SVM 分类 5.5）。
SVR shines on small-to-medium nonlinear data but scales poorly; know the kernel trick (it transfers to SVM classification).


<a id="7"></a>
## 7. 小结 / Summary

```
ε-不敏感损失: 管道内(|误差|≤ε)零惩罚, 管道外线性惩罚
  → 稀疏(只支持向量定义模型) + 稳健(线性非平方)
核技巧 ⭐: 隐式映射到(无限维)高维, 只算核 K(xi,xj)=内积, 不算坐标
  RBF 核 e^{-γ‖xi-xj‖²} = 无限维特征空间
三超参: C(违规惩罚) / ε(管道宽度) / gamma(影响范围, RBF)
支持向量: 边界上/外的点; 管道内点删了模型不变
缺点: 大样本慢 O(n²~n³), 不可解释; 必须标准化
```

### 💡 面试速查
1. **ε-不敏感损失**: 管道内零惩罚 → 稀疏 + 稳健
2. **核技巧**: 不显式升维, 只算核函数(内积) → 隐式无限维线性回归
3. **支持向量**: 只有边界上/外的点决定模型
4. **gamma 大→过拟合**(影响范围小, 曲线扭曲)
5. **SVR 大样本慢** → 树/NN 取代; 但核技巧思想重要(→SVM 5.5)

### 下一节
**4.10 KNN 回归**——SVR 是参数模型(学权重)。KNN 是极致的**非参数/惰性**模型: 不训练, 预测时直接看最近邻的平均。最简单的非线性回归。
